In [1]:
import numpy as np
import pandas as pd
import json
import pyodbc
import config

In [2]:
def get_lims_connection():
    global odbcLimsConnection
    # odbcLimsConnection = oracledb.connect(user=config.limsdb_username, password=config.limsdb_password,
    # dsn=config.limsdb_dsn)
    odbcLimsConnection = pyodbc.connect('DRIVER=' + config.limsdb_driver + ';DATABASE=' + config.limsdb_database +
                                        ';UID=' + config.limsdb_username + ';PWD=' + config.limsdb_password +
                                        ';SERVER=' + config.limsdb_hostname + ';PORT=' + config.limsdb_port +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection


if __name__ == '__main__':
    try:  # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_lims_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)

In [3]:
column_names = ['sample_number','audit_timestamp','audit_type','trans_string']
sql_command = "select distinct SAMPLE_NUMBER, AUDIT_TIMESTAMP, AUDIT_TYPE, TRANS_STRING from SAMPLE_AUDIT_DECRYPT"
with odbcLimsConnection.cursor() as lims_cursor_fetch_all:
    lims_cursor_fetch_all.execute(sql_command)
    queryFetch = lims_cursor_fetch_all.fetchall()
dataFrame = pd.DataFrame.from_records(queryFetch, columns=column_names)

In [5]:
dataFrame['trans_string'].apply(json.loads).to_json(path_or_buf="audit.json", orient="records")
with open("audit.json") as jsonEntries:
    jsonDataFrame = json.load(jsonEntries)
jsonDataFrame = pd.json_normalize(jsonDataFrame)

In [7]:
dataFrame = dataFrame.drop(columns=['trans_string'])
jsonDataFrame = jsonDataFrame.replace(np.nan, "█").replace("▄", ",", regex=True)
encodedDataFrame = pd.concat([dataFrame, jsonDataFrame], axis=1)

In [8]:
encodedDataFrame

,sample_number,audit_timestamp,audit_type,SAMPLE_NUMBER,TEST_NUMBER,ORDER_NUMBER,ANALYSIS,VERSION,ANALYSIS_COUNT,GROUP_NAME,...,ENTERED_BY,ENTRY_TYPE,RECD_DATE,RECEIVED_BY,DATE_RECEIVED,NUMERIC_ENTRY,CONTROL_1,DESCRIPTION,CONTAINER_TYPE,T_CONTAINER_UNITS
0,8331,2023-09-06 18:20:12,LogSample,8331,20177,1,ID,1,0,DEFAULT,...,█,█,█,█,█,█,█,█,█,█
1,8331,2023-09-06 18:20:12,LogSample,8331,20177,1,ID,█,█,█,...,█,█,█,█,█,█,█,█,█,█
2,8331,2023-09-06 18:20:12,LogSample,8331,20178,2,APPEARANCE,3,0,DEFAULT,...,█,█,█,█,█,█,█,█,█,█
3,8331,2023-09-06 18:20:12,LogSample,8331,20178,1,APPEARANCE,█,█,█,...,█,█,█,█,█,█,█,█,█,█
4,8331,2023-09-06 18:20:12,LogSample,8331,█,█,█,█,█,DEMO,...,█,█,█,█,█,█,█,█,█,█
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882629,28254,2023-09-07 04:32:46,CalculationUpdate,28254,█,█,█,█,█,█,...,█,█,█,█,█,█,█,█,█,█
882630,28254,2023-09-07 04:32:46,CalculationUpdate,█,50038,█,PKG_APPEARANCE,█,█,█,...,█,█,█,█,█,█,█,█,█,█
882631,28254,2023-09-07 04:32:46,CalculationUpdate,█,50038,█,PKG_APPEARANCE,█,█,█,...,█,█,█,█,█,█,█,█,█,█
882632,28254,2023-09-07 04:32:46,CalculationUpdate,█,50038,█,PKG_APPEARANCE,█,█,█,...,█,█,█,█,█,█,█,█,█,█
